# ifuCube: the Helix Nebula

A short tour of the package on real data. Download one LVM pointing of
NGC 7293, build emission-line maps from it, confirm no flux was lost, make a
Balmer-decrement map and a velocity map, then build a cube.

Everything runs on the public LVM preview released with SDSS DR20. Section 2
fetches the data, so there is nothing to arrange beforehand.

## 1. Setup

In [ ]:
# Make the package importable however this notebook was launched. Jupyter sets
# the working directory to the notebook's own folder, so ifuCube sits one level
# up; walking up from here also works from the repo root or a copy saved deeper
# in the tree. A no-op if you ran `pip install -e .`.
import sys
from pathlib import Path

try:
    import ifuCube
except ModuleNotFoundError:
    here = Path.cwd().resolve()
    for parent in (here, *here.parents):
        if (parent / "ifuCube" / "__init__.py").exists():
            sys.path.insert(0, str(parent))
            break
    else:
        raise ModuleNotFoundError(
            f"could not find the ifuCube package above {here}. Run this from "
            "inside the repository, or `pip install -e .` from the repo root.")
    import ifuCube

import numpy as np
import matplotlib.pyplot as plt

print(f"ifuCube {ifuCube.__version__}")

Three numbers set the output grid, and two of them matter more than they look.

`fwhm_arcsec` is the width of the kernel each fibre's flux is spread with. Set
it from the **fibre spacing** (37″ for LVM), not from the pixel size: a kernel
narrower than the spacing leaves gaps between fibres however fine the grid is.

`pixel_size_arcsec` at 9″ puts about four pixels across the ~35″ resolution
element, which is comfortably Nyquist. File size grows as the *square* of this,
so 2″ pixels cost roughly twenty times the bytes and carry no more information.

In [ ]:
PIXEL = 9.0           # arcsec, output spaxel
FWHM = 35.0           # arcsec, kernel width -- from the fibre spacing
FIBER_RADIUS = 17.65  # arcsec, = 35.3 / 2

OUT = Path("mapcats")
OUT.mkdir(exist_ok=True)

## 2. Getting the data

The Helix is one of six regions in the LVM preview in **SDSS DR20**, and it is
public — no account needed. The pointing analysed in the LVM Helix paper
(Sánchez et al. 2026) is tile **11111**, MJD **60191**, exposure **4297**.

`sdss_access` builds the Science Archive Server paths from those numbers, which
is less error-prone than keeping a URL correct by hand — it even derives the
tile group `0011XX` from the tile ID.

In [ ]:
# pip install sdss-access
from sdss_access import Access

DRPVER, DAPVER = "1.2.0", "1.2.0.251218"
HELIX = dict(tileid=11111, mjd=60191, expnum=4297)

access = Access(release="DR20")
dap_kw = dict(drpver=DRPVER, dapver=DAPVER, rspid="rsp108",
              snlevel="sn20", daptype="dap", **HELIX)

# Look before downloading: these are the URLs and the local destinations.
print("DAP   ", access.url("lvm_dap", **dap_kw))
print("SFrame", access.url("lvm_sframe", drpver=DRPVER, **HELIX))

In [ ]:
access.remote()
access.add("lvm_dap", **dap_kw)
access.add("lvm_sframe", drpver=DRPVER, **HELIX)
access.set_stream()
access.commit()

DAP = Path(access.full("lvm_dap", **dap_kw))
SFRAME = Path(access.full("lvm_sframe", drpver=DRPVER, **HELIX))
print("DAP   :", DAP.exists(), DAP)
print("SFrame:", SFRAME.exists(), SFRAME)

DR20 holds 169 reduced exposures and more than one may cover the Helix. The
summary tables are the index — `dapall` lists every analysed exposure — so you
can select by position rather than trusting a number copied from a paper. Worth
doing, since that paper describes the DR19 processing of this field.

In [ ]:
from astropy.table import Table
from astropy.coordinates import SkyCoord
from astropy.utils.data import download_file
import astropy.units as u

SAS = "https://data.sdss.org/sas/dr20/spectro/lvm"
dapall = Table.read(download_file(
    f"{SAS}/analysis/{DRPVER}/{DAPVER}/dapall-{DRPVER}-{DAPVER}.fits",
    cache=True))

helix = SkyCoord(337.4108 * u.deg, -20.8372 * u.deg)   # NGC 7293

# Column naming shifts between releases, so find the pair rather than assume it
ra_col = next((c for c in dapall.colnames if c.lower() in ("ra", "ifura", "plug_ra")), None)
dec_col = next((c for c in dapall.colnames if c.lower() in ("dec", "ifudec", "plug_dec")), None)

if ra_col and dec_col:
    sep = helix.separation(SkyCoord(dapall[ra_col] * u.deg, dapall[dec_col] * u.deg))
    near = dapall[sep < 1 * u.deg]
    near["sep_arcmin"] = sep[sep < 1 * u.deg].to(u.arcmin).round(1)
    cols = [c for c in ("tileid", "mjd", "expnum", "sep_arcmin") if c in near.colnames]
    print(f"{len(near)} exposures within 1 deg of the Helix:")
    print(near[cols])

## 3. Read the fibres

`read_dap_multi` returns a `FiberSet`: fluxes, inverse variances, wavelengths,
fibre positions, and a metadata dictionary holding the per-fibre catalogue.

It decides which fibres are science fibres and which are healthy **from the
file itself**, rather than from an external fibre map that may have changed
since the exposure, and it records every fibre as kept or rejected so a map can
be traced back to the mask that produced it.

Reading DAP files needs **`lvmdap`**, which is not on PyPI and so is not pulled
in by `pip install -e .`:

```
pip install --no-deps git+https://github.com/sdss/lvmdap.git
```

Use `--no-deps`: lvmdap pins `matplotlib==3.7.3` exactly and pulls pyPipe3D,
specutils, emcee, PyAstronomy and lmfit, none of which the file reader needs.
This is the only part of ifuCube that requires it.

In [ ]:
from ifuCube.io_lvm import read_dap_multi, list_dap_columns

# What is in this file? Useful the first time, and when a release renames
# something.
list_dap_columns(DAP, "*Halpha*")

In [ ]:
fib = read_dap_multi(DAP, slitmap_path=DAP, verbose=True)

print()
print("lines       :", fib.meta["line_names"])
print("fibres used :", fib.flux.shape[0])

## 4. Build the maps

Each fibre's flux is spread over nearby pixels with a Gaussian kernel, and all
the weights go into one sparse matrix **A** of shape (pixels, fibres). The
columns of **A** sum to exactly 1, so each fibre hands out all of its flux and
none is lost.

That single property is the whole design. It makes the map *flux-conserving*:
summing the pixels of a region gives the total flux in that region, which is
what you want for photometry.

It also means this is a **deposition, not an interpolation**. A pixel's value
answers "how much flux was assigned here", not "what is the brightness at this
point" — a distinction that matters in the next section but one.

In [ ]:
from ifuCube.mapcat_build import build_map

res = build_map(fib,
                pixel_size_arcsec=PIXEL,
                fwhm_arcsec=FWHM,
                fiber_radius_arcsec=FIBER_RADIUS)

print("grid :", res["inside"].shape)
print("A    :", res["A"].shape, f"({res['A'].nnz:,} non-zero)")

## 5. Confirm nothing was lost

Worth running after any parameter change — it is cheap, and it is what catches
a mis-specified grid.

In [ ]:
from ifuCube.diagnostics import run_all_checks

run_all_checks(res)

Checks 1, 2 and 4 are the real tests: total flux out over in, the operator
against its adjoint, and every column of **A** summing to one.

**Check 3 is informational, not pass/fail.** It compares each fibre's flux with
a weighted average of its neighbourhood, which is exact only for a constant
field — so its spread measures structure in the *source*, not flux loss. On a
flat field it is zero; on a ring like the Helix, around 0.3.

Because the projection conserves flux, integrated photometry is just a sum:

In [ ]:
i = {n: k for k, n in enumerate(res["line_names"])}
ha = res["map"][i["Halpha"]]

print(f"sum over the map   : {np.nansum(ha[res['inside']]):.6g}")
print(f"sum over the fibres: {np.nansum(fib.flux[:, i['Halpha']]):.6g}")

## 6. Look at the maps

Maps come out North up and East left, the standard sky orientation.

Index planes **by name**. Reorder the `lines` argument once and positional
indexing hands you an H-alpha map labelled H-beta, which looks entirely
plausible.

In [ ]:
from ifuCube.plotting import show_map, show_planes

show_map(ha, wcs=res["wcs2d"], mask=res["inside"], log=True,
         title="Helix, H-alpha", label="flux / pixel")
plt.show()

In [ ]:
want = [n for n in ("Halpha", "Hbeta", "OIII5007", "NII6583") if n in i]
show_planes(np.stack([res["map"][i[n]] for n in want]), want,
            wcs=res["wcs2d"], mask=res["inside"], ncols=2, log=True,
            label="flux / pixel")
plt.show()

## 7. Line ratios

Ratios are the easiest thing here, because **everything cancels**. Both lines go
through the same **A**, so the ratio of two stored planes is already correct:
pixel area and the fibre aperture fill factor divide out algebraically. No
corrections, no normalisation.

What does not cancel is noise. H-beta is the faint line, and a ratio diverges
where its denominator falls into the noise — so cut on signal-to-noise in the
weak line.

In [ ]:
hb = res["map"][i["Hbeta"]]
snr_hb = np.where(res["ivar"][i["Hbeta"]] > 0,
                  hb * np.sqrt(res["ivar"][i["Hbeta"]]), 0.0)

ok = res["inside"] & (snr_hb > 3) & (hb > 0)
ratio = np.where(ok, ha / hb, np.nan)

print(f"S/N(Hbeta) > 3 over {100 * ok.sum() / res['inside'].sum():.0f}% of the field")
print(f"Halpha/Hbeta 16/50/84%: {np.nanpercentile(ratio[ok], [16, 50, 84]).round(2)}")

In [ ]:
# Case B recombination gives an intrinsic ratio of 2.86, so
# E(B-V) = 2.32 * log10(R / 2.86).
#
# Read this loosely for the Helix: a thin expanding planetary nebula is not the
# dusty H II region that assumption is built for, and in a PN the Balmer ratio
# carries temperature and density information as well as dust.
show_map(ratio, wcs=res["wcs2d"], title="Halpha / Hbeta", label="ratio",
         cmap="viridis", pct=(2, 98))
plt.show()

## 8. Kinematics need a different operator

Flux is **extensive** — it adds up, so it is deposited with column-normalised
weights and totals are preserved.

Velocity is **intensive** — it does not add up. Depositing it the same way would
give a pixel the *sum* of its neighbours' velocities, so a region uniformly at
−27 km/s with three contributing fibres would read −81 km/s. The right operator
is a weighted average over the same matrix:

$$v_p = \frac{\sum_k A_{pk}\,w_k\,v_k}{\sum_k A_{pk}\,w_k}$$

Bad fibres are handled differently too. `build_map` scrubs bad fluxes to 0.0,
which is harmless because zero flux contributes zero. Here it would be a
disaster: **0 km/s is a valid velocity**, so a dead fibre has its *weight*
zeroed, not its value.

`build_kinematic_map` takes the result of `build_map` and reuses its **A**, so
the velocity map lands on exactly the same grid and WCS as the flux map.

In [ ]:
from ifuCube.mapcat_build import build_kinematic_map

kin = build_kinematic_map(res, quantity="vel", weighting="ivar")
v = kin["map"][0]

print(f"\nmedian velocity: {np.nanmedian(v[np.isfinite(v)]):+.1f} km/s")
print("NGC 7293 systemic velocity is about -27 km/s")

In [ ]:
show_map(v, wcs=res["wcs2d"], mask=res["inside"] & np.isfinite(v),
         title="Helix, H-alpha velocity", label="km/s",
         cmap="RdBu_r", pct=(5, 95))
plt.show()

One unit trap: the DAP's `disp_*` columns are line widths in **Ångström**, not
km/s. Convert with $\sigma_v = c\,\sigma_\lambda/\lambda$ and subtract the
instrumental width in quadrature — at LVM's R ≈ 4000 that is about 0.70 Å, or
32 km/s at H-alpha, so most lines here are unresolved.

## 9. Save and read back

Maps are stored as flux per pixel, the conserving quantity. `read_map` converts
to surface brightness on the way in, so maps built with different pixel sizes
are directly comparable. It also hands back an `index()` so nothing downstream
has to count planes.

In [ ]:
from ifuCube.mapcat_build import save_map_to_fits, read_map

save_map_to_fits(res, OUT / "helix_map.fits")
save_map_to_fits(kin, OUT / "helix_vel.fits")

m = read_map(OUT / "helix_map.fits")
print("units       :", m["bunit"])
print("lines       :", m["line_names"])
print("index(Hbeta):", m["index"]("Hbeta"))
print("FIBERCAT    :", len(m["fibercat"]), "rows")

## 10. A cube

The same operator over every wavelength plane instead of a handful of fitted
lines. This uses the reduced spectra rather than the DAP file.

Cubes are stored as **surface brightness**, since those values do not depend on
the grid the cube was built with. Size scales as the pixel size squared: at 9″ a
full pointing is about 3.3 GB for flux and inverse variance together.
`inplace=True` converts without allocating a second cube, which is what you want
whenever nothing reads the array afterwards.

In [ ]:
from ifuCube.io_lvm import read_lvm_sframe
from ifuCube.cube_build import build_cube, save_cube_to_fits, read_cube

Path("cubes").mkdir(exist_ok=True)
cfib = read_lvm_sframe(SFRAME, verbose=True)
cube = build_cube(cfib, pixel_size_arcsec=PIXEL, fwhm_arcsec=FWHM,
                  fiber_radius_arcsec=FIBER_RADIUS, chunk_mb=256.0)
save_cube_to_fits(cube, "cubes/helix_cube.fits", inplace=True)

Read a **window**, not the whole cube — `wave_range` memmaps and slices, so an
H-alpha window costs a few MB instead of gigabytes.

In [ ]:
from ifuCube.plotting import show_wave

cc = read_cube("cubes/helix_cube.fits", wave_range=(6540, 6590))
print("window:", cc["data"].shape, "|", cc["bunit"])

# width averages over a few Angstrom; a single plane is noisy
show_wave("cubes/helix_cube.fits", [6548., 6563., 6583.], width=3.0,
          ncols=3, cmap="inferno")
plt.show()

## Where to go next

- `ifuCube.ratio` — ratio maps with propagated errors, written to FITS
- `ifuCube.regions` — select fibres by circle or polygon and coadd their spectra
- `ifuCube.mosaic` — combine overlapping pointings onto a common grid
- `ifuCube.healpix_build` — move maps onto a HEALPix grid, merge and plot
- `ifuCube.fill` — estimate rejected fibres from their neighbours, for a
  gap-free figure

One caution for statistics. At 9″ pixels a 35″ kernel puts each fibre on
roughly fifteen pixels, so a per-pixel scatter plot — a BPT diagram, say — shows
every measurement about fifteen times over, with neighbouring points correlated
because they share fibres. Anything fitted to that is overconfident by about
√15. Use the `FIBERCAT` table, which has one independent row per measurement;
the map is for showing *where*.

Both builders also run from the command line, which is what you want for a
batch of exposures:

    python -m ifuCube.mapcat_build --dap FILE -o mapcats/x.fits --pixel 9 \
        --kinematics vel --check
    python -m ifuCube.cube_build --sframe FILE -o cubes/x.fits --pixel 9

Add `--help` to either for the full list of options.